In [1]:
from proj_utils import *

import os
import pickle
from os.path import join
from collections import defaultdict, Counter
from itertools import combinations
from datetime import datetime
from dateutil.relativedelta import relativedelta
import ipywidgets as widgets
from IPython.display import display

import numpy as np
import pandas as pd
from tqdm import tqdm

from scipy.optimize import linear_sum_assignment
from scipy.optimize import curve_fit
from sklearn.metrics.pairwise import cosine_similarity

import torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import ipywidgets as widgets
from ipywidgets import interact, interact_manual


import bertopic
from sentence_transformers import SentenceTransformer

from proj_utils import *

/home/seungwoong.ha/anaconda3/envs/collmind/lib/python3.11/site-packages/umap/distances.py:1063: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/seungwoong.ha/anaconda3/envs/collmind/lib/python3.11/site-packages/umap/distances.py:1071: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/seungwoong.ha/anaconda3/envs/collmind/lib/pyth

In [2]:
collection_name = 'Motherjones'
model_name = MODEL_NAMES[collection_name]
threshold = 10

start_date, end_date = DATE_RANGES[collection_name]
#topic_model = (BERTopic.load(join('model', collection_name.lower(), model_name), embedding_model="all-MiniLM-L6-v2"))

In [3]:
df = pd.read_csv(join('ctfidf', collection_name.lower(), 'topics_per_month.csv'))
total_frequency = df.groupby('Month').sum()['Frequency']
df['norm_freq'] = df.apply(lambda row: row['Frequency'] / total_frequency[row['Month']], axis=1)
df['rank'] = df.groupby('Month')['norm_freq'].rank(ascending=False)
topic_num = len(df['Topic'].unique())-1 # excluding -1

df2 = df[df['Topic'] != -1]
total_frequency2 = df2.groupby('Month').sum()['Frequency']
df2['norm_freq'] = df.apply(lambda row: row['Frequency'] / total_frequency2[row['Month']], axis=1)
df2['rank'] = df2.groupby('Month')['norm_freq'].rank(ascending=False)

In [4]:
# load articles_by_month from pickle file
with open(join('article', collection_name.lower(), model_name, 'articles_by_month.pkl'), 'rb') as f:
    articles_by_month = pickle.load(f)
    
# remove articles with less than threshold comments
for month, articles in articles_by_month.items():
    articles_by_month[month] = articles[articles['comment_id'].apply(len) > threshold]

In [5]:
# load total_mean_embedding_list  and total_nonexist_index_list from pickle
with open(join('transform', collection_name.lower(), model_name, 'total_mean_embedding_list.pickle'), 'rb') as f:
    total_mean_embedding_list = pickle.load(f)
with open(join('transform', collection_name.lower(), model_name, 'total_nonexist_index_list.pickle'), 'rb') as f:
    total_nonexist_index_list = pickle.load(f)

### Figure 1.